# QuantProof walkthrough

Run from the repository root after `pip install -e ".[all]"`. Everything uses the bundled
synthetic data, so the notebook works offline.

In [ ]:
from pathlib import Path

import pandas as pd

from quantproof import AuditConfig, audit
from quantproof.reports import render_markdown

ROOT = Path.cwd() if (Path.cwd() / "examples").exists() else Path.cwd().parents[1]
prices = pd.read_parquet(ROOT / "examples/data/prices.parquet")
prices.tail()

## 1. Audit a strategy with look-ahead bias

Static analysis, the future-perturbation test and the value-level leakage check each find
the problem independently.

In [ ]:
result = audit(
    strategy=ROOT / "examples/lookahead_strategy/strategy.py",
    data=prices,
    config=AuditConfig(profile="quick"),
)
print(result.status)
pd.DataFrame(
    [(f.severity.value, f.id, f.title) for f in result.issues],
    columns=["severity", "rule", "title"],
)

## 2. The evidence behind a finding

Every finding carries machine-readable evidence; here, the perturbation trials that changed
past decisions.

In [ ]:
causal = next(f for f in result.findings if f.id == "QP-CAUSAL-001")
pd.DataFrame(causal.evidence["examples"])

## 3. Selection bias: the overfit example

In [ ]:
overfit = audit(
    strategy=ROOT / "examples/overfit_strategy/strategy.py",
    data=ROOT / "examples/data/noise.parquet",
    config=AuditConfig(profile="quick"),
)
stats, val = overfit.sections["statistics"], overfit.sections["validation"]
{
    "naive Sharpe": overfit.sections["overview"]["headline"]["sharpe"],
    "Deflated Sharpe Ratio": stats["dsr"]["dsr"],
    "trials": stats["dsr"]["n_trials"],
    "PBO": val["pbo"]["pbo"],
    "walk-forward OOS Sharpe": val["walk_forward"]["oos_sharpe"],
}

## 4. Building blocks on their own

In [ ]:
import numpy as np

from quantproof.statistics import deflated_sharpe_ratio, probabilistic_sharpe_ratio
from quantproof.validation import CPCV, PurgedKFold

returns = prices["close"].pct_change().dropna()
print(probabilistic_sharpe_ratio(returns).psr, deflated_sharpe_ratio(returns, n_trials=50).dsr)

cv = CPCV(n_groups=6, n_test_groups=2, embargo=0.01)
print(cv.n_splits, "splits,", cv.n_paths, "paths")
for train, test in PurgedKFold(5, label_horizon=5, embargo=0.01).split(np.zeros(len(returns))):
    print(len(train), "train /", len(test), "test")

## 5. A Markdown report for a pull request

In [ ]:
print(render_markdown(result)[:1500])